# Export Fine-Tuned Gemma 2 for Ollama (Kaggle Version)

**What this does:** Merges your LoRA adapter into the base Gemma 2 2B model and converts it to a GGUF file for Ollama.

### Before you start:
1. **Enable GPU:** Settings (right panel) → Accelerator → GPU T4 x2
2. **Enable Internet:** Settings → Internet → On
3. **Upload your adapter:** Click "+ Add Input" → Upload → upload the `mi-therapy-gemma2-v2` folder as a Kaggle Dataset

### How to upload adapter as Kaggle Dataset:
1. Go to kaggle.com → Datasets → New Dataset
2. Name it `mi-therapy-adapter-v2`
3. Upload ALL files from your Google Drive folder: `mi-therapy-capstone/adapters/mi-therapy-gemma2-v2/`
   - `adapter_config.json`
   - `adapter_model.safetensors`
   - `tokenizer.json`, `tokenizer_config.json`, `tokenizer.model`
   - `special_tokens_map.json`
   - Any other files in that folder
4. Click Create
5. Then in this notebook: "+ Add Input" → search `mi-therapy-adapter-v2` → Add

Your adapter will be at: `/kaggle/input/mi-therapy-adapter-v2/`

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers>=4.44.0 peft>=0.12.0 accelerate>=0.33.0 bitsandbytes
!pip install -q gguf sentencepiece protobuf

print('\n✅ Dependencies installed')

## Step 2: Verify Adapter Files & GPU

In [ ]:
import torch, os

# GPU check
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    raise RuntimeError('❌ No GPU! Go to Settings → Accelerator → GPU T4 x2')

# Adapter check
ADAPTER_PATH = '/kaggle/input/mi-therapy-adapter-v2'

# If you named your dataset differently, update the path:
# ADAPTER_PATH = '/kaggle/input/YOUR-DATASET-NAME'

print(f'\nLooking for adapter at: {ADAPTER_PATH}')
if os.path.exists(ADAPTER_PATH):
    files = os.listdir(ADAPTER_PATH)
    print(f'✅ Found {len(files)} files:')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(ADAPTER_PATH, f))
        print(f'   {f} ({size/1024:.1f} KB)' if size < 1024*1024 else f'   {f} ({size/1024/1024:.1f} MB)')
    
    # Check for required file
    if 'adapter_config.json' not in files:
        # Maybe files are in a subfolder
        for sub in files:
            subpath = os.path.join(ADAPTER_PATH, sub)
            if os.path.isdir(subpath) and 'adapter_config.json' in os.listdir(subpath):
                ADAPTER_PATH = subpath
                print(f'\n✅ Adapter found in subfolder: {ADAPTER_PATH}')
                break
        else:
            print('\n⚠️  adapter_config.json not found! Make sure you uploaded all adapter files.')
else:
    print('❌ Adapter not found!')
    print('   1. Click "+ Add Input" in the right panel')
    print('   2. Search for your uploaded dataset')
    print('   3. Add it, then re-run this cell')
    # List what IS available
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        print(f'\n   Available inputs: {os.listdir(input_dir)}')

## Step 3: Load Base Model + Merge LoRA Adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'google/gemma-2-2b-it'
MERGED_PATH = '/kaggle/working/gemma2-mi-merged'

print('Loading base model (this takes 1-2 minutes)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto'
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f'Loading LoRA adapter from {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print('Merging LoRA weights into base model...')
merged_model = model.merge_and_unload()

print(f'Saving merged model to {MERGED_PATH}...')
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f'\n✅ Merge complete!')
print(f'   GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## Step 4: Quick Sanity Test

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=merged_model, tokenizer=tokenizer, max_new_tokens=150)

# Test: High defensiveness client
test_messages = [
    {'role': 'user', 'content': "I don't think I have a problem with drinking. Everyone does it."}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, do_sample=True, temperature=0.7, top_p=0.9)

print('Test prompt: "I don\'t think I have a problem with drinking. Everyone does it."')
print(f'\nModel response:')
print(output[0]['generated_text'][len(prompt):])
print('\n✅ If the response sounds like an MI therapist, the merge worked!')

## Step 5: Convert to GGUF Format

This creates a quantized GGUF file (~1.5GB) that Ollama can load.

In [ ]:
# Clone llama.cpp for conversion
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

print('\n✅ llama.cpp ready')

In [ ]:
# Convert merged model to GGUF (Q4_K_M quantization — best balance of quality vs size)
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/gemma2-mi-merged \
    --outfile /kaggle/working/gemma2-mi-therapist.gguf \
    --outtype q4_K_M

import os
gguf_path = '/kaggle/working/gemma2-mi-therapist.gguf'
size_gb = os.path.getsize(gguf_path) / (1024**3)
print(f'\n✅ GGUF created: {size_gb:.2f} GB')
print(f'   Path: {gguf_path}')

## Step 6: Download the GGUF File

**Option A (easiest):** The GGUF is in `/kaggle/working/` — when you click **"Save Version"** → **"Save & Run All"**, it becomes a Kaggle Output you can download from the notebook's Output tab.

**Option B:** Run the cell below to get a direct download link.

In [ ]:
# Option B: Download directly from notebook
from IPython.display import FileLink
import shutil

# Move to output directory so it appears in Kaggle's Output tab
output_path = '/kaggle/working/gemma2-mi-therapist.gguf'
print(f'GGUF file: {output_path}')
print(f'Size: {os.path.getsize(output_path)/1024**3:.2f} GB')
print()
print('=== HOW TO DOWNLOAD ===')
print('1. Click "Save Version" (top right) → Quick Save')
print('2. After it finishes, go to your notebook page')
print('3. Click the "Output" tab')
print('4. Download gemma2-mi-therapist.gguf')
print()
print('=== THEN ON YOUR LAPTOP ===')
print('1. Install Ollama from https://ollama.com')
print('2. Put the GGUF file and Modelfile in the same folder')
print('3. Run: ollama create mi-therapist -f Modelfile')
print('4. Test: ollama run mi-therapist "Hello"')
print('5. Update .env: OLLAMA_MODEL=mi-therapist')

## After Download: Ollama Setup (Run on Your Laptop)

### 1. Install Ollama
Download from https://ollama.com and install it.

### 2. Create a Modelfile
In the same folder as `gemma2-mi-therapist.gguf`, create a file called `Modelfile`:

```
FROM ./gemma2-mi-therapist.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.2
PARAMETER num_predict 200

TEMPLATE """<start_of_turn>user
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
"""
```

### 3. Create the Ollama model
```bash
ollama create mi-therapist -f Modelfile
```

### 4. Test it
```bash
ollama run mi-therapist "I don't think I have a problem with drinking."
```

### 5. Update your chatbot
In `.env`:
```
OLLAMA_MODEL=mi-therapist
```

Then run `streamlit run app.py` — your chatbot now uses YOUR fine-tuned model!